<a href="https://colab.research.google.com/github/chiaranovelli/Magdeburg_school/blob/main/PCA_POD_ex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PCA and POD on a Laminar Coflow Flame

**Author:** Chiara Novelli, Prof. Alberto Procacci

**Email:** chiara.novelli@ulb.be, alberto.procacci@ulb.be


## Introduction

This notebook applies **Principal Component Analysis (PCA)** and **Proper Orthogonal Decomposition (POD)** to a dataset obtained from numerical simulations of a laminar, axisymmetric, non-premixed coflow flame.

 The fuel stream is a nitrogen-diluted CH₄/N₂ mixture (65% CH₄, 35% N₂ molar), and the oxidizer is air, both at ambient temperature and atmospheric pressure.

The dataset is stored as a **3D tensor**:

$$\mathbf{X} \in \mathbb{R}^{n_{\mathrm{f}} \times n_{\mathrm{s}} \times n_{\mathrm{t}}}$$

with:
- $n_{\mathrm{f}} = 10$: T, O, O₂, OH, H₂O, CH₄, CO, CO₂, C₂H₂, N₂
- $n_r = 68$, $n_z = 200$:  $n_{\mathrm{s}} = n_r\,*\,n_z =  13{,}600$ spatial points
- $n_{\mathrm{t}} = 2001$ snapshots

The notebook is structured as follows:

1. **Loading and visualising the data**
2. **PCA on a single snapshot** — relationships between thermochemical variables
3. **Proper Orthogonal Decomposition (POD)** — energy-ranked spatial modes and temporal scores

#  1: Loading and visualising the data

First dowload the dataset

In [ ]:
!pip install -q gdown

file_id = "1S4z6zn2KZ97LCTFMahyKh7GGMnKHfcf6"
out_name = "X.npy"
!gdown --id {file_id} -O {out_name}

import numpy as np
data = np.load(out_name)
print('data loaded successfully')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X_full   = data['data_3D']          # (n_f, n_s, n_t)
r_coords = data['x_coords_flat']    # radial coordinate [m], shape (n_s,)
z_coords = data['z_coords_flat']    # axial coordinate  [m], shape (n_s,)
features = [str(f) for f in data['features']]   # ['T','O','O2','OH','H2O','CH4','CO','CO2','C2H2','N2']
shape_info = data['shape_info']     # [n_f, nr, nz, n_t]

n_f, n_s, n_t = X_full.shape
nr, nz = shape_info[1], shape_info[2]

print(f'Features  : {features}')
print(f'Grid      : nr={nr}, nz={nz}')
print(f'Snapshots : n_t={n_t}')
print(f'Data shape: {X_full.shape}  (n_f, n_s, n_t)')


Before any decomposition, let's visualise a snapshot to get a physical sense of the data.

In [ ]:
import matplotlib.tri as tri

# ── Triangulation  ──────────────────────────────────────
triang = tri.Triangulation(r_coords, z_coords)
xy_tri = np.column_stack([r_coords, z_coords])
tris   = triang.triangles
edges  = [np.linalg.norm(xy_tri[tris[:, i]] - xy_tri[tris[:, (i+1)%3]], axis=1) for i in range(3)]
triang.set_mask(np.max(edges, axis=0) > 5 * np.median(np.max(edges, axis=0)))

triang_cm = tri.Triangulation(r_coords * 1e2, z_coords * 1e2, triang.triangles)
triang_cm.set_mask(triang.mask)

r_max_cm = r_coords.max() * 1e2
z_min_cm = z_coords.min() * 1e2
z_max_cm = 10.0

# ── Colormaps and labels for all features ────────────────────────────────────
CMAPS = {'T':    'inferno',  'OH':   'YlOrRd', 'CH4': 'Blues',
         'O2':   'Blues',    'CO':   'Greens',  'CO2': 'copper_r',
         'H2O':  'YlGnBu',  'N2':   'viridis', 'O':   'Oranges',
         'C2H2': 'Purples'}
LABELS = {'T':    'T [K]',
          'OH':   r'$Y_\mathrm{OH}$  [-]',
          'CH4':  r'$Y_\mathrm{CH_4}$  [-]',
          'O2':   r'$Y_\mathrm{O_2}$  [-]',
          'CO':   r'$Y_\mathrm{CO}$  [-]',
          'CO2':  r'$Y_\mathrm{CO_2}$  [-]',
          'H2O':  r'$Y_\mathrm{H_2O}$  [-]',
          'N2':   r'$Y_\mathrm{N_2}$  [-]',
          'O':    r'$Y_\mathrm{O}$  [-]',
          'C2H2': r'$Y_\mathrm{C_2H_2}$  [-]'}

# ── Single-feature plot ───────────────────────────────────────────────
def plot_feature(feat, t_snap, ax=None, show_ylabel=True):
    """
    Plot one feature at snapshot t_snap.
    Pass ax for multi-panel use; omit to get a standalone figure.
    """
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(2.0, 7.0), constrained_layout=True)

    field = X_full[features.index(feat), :, t_snap]
    cf    = ax.tricontourf(triang_cm, field, levels=60,
                           cmap=CMAPS.get(feat, 'viridis'))
    cb    = ax.get_figure().colorbar(cf, ax=ax, shrink=0.7, pad=0.03)
    cb.set_label(LABELS.get(feat, feat), fontsize=10)
    cb.ax.tick_params(labelsize=8)

    ax.set_xlim([0, r_max_cm])
    ax.set_ylim([z_min_cm, z_max_cm])
    ax.set_aspect('equal')
    ax.set_xlabel('r  [cm]', fontsize=10)
    if show_ylabel:
        ax.set_ylabel('z  [cm]', fontsize=10)
    else:
        ax.set_yticklabels([])
    ax.tick_params(labelsize=8)
    ax.set_title(feat, fontsize=11, fontweight='bold', pad=6)
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

    if standalone:
        plt.show()

# plotting

t_snap = 2000
features_to_plot = 'T'
plot_feature(features_to_plot, t_snap)


Now let's see the full simulation evolving in time:

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML

DT_SAVE = 2e-3

# ── Exact layout (inches) — preserves equal aspect without white space ────────
W_axes = 0.45
aspect = (z_max_cm - z_min_cm) / r_max_cm
H_axes = W_axes * aspect

m_l, m_b, m_t     = 0.45, 0.32, 0.28
cb_gap, cb_w, m_r = 0.07, 0.10, 0.42

_fig_w = m_l + W_axes + cb_gap + cb_w + m_r
_fig_h = m_t + H_axes + m_b

def _n(x, tot): return x / tot

ax_rect = [_n(m_l,                    _fig_w), _n(m_b, _fig_h),
           _n(W_axes,                 _fig_w), _n(H_axes, _fig_h)]
cb_rect = [_n(m_l + W_axes + cb_gap, _fig_w), _n(m_b, _fig_h),
           _n(cb_w,                  _fig_w), _n(H_axes, _fig_h)]

FS = 7

def animate_feature(feat, t_start=0, t_end=None, step=10,
                    dynamic_clim=False, p_low=1, p_high=99):
    """Inline jshtml animation of one feature. Uses triang_cm, CMAPS, LABELS from cell above."""
    if feat not in features:
        raise ValueError(f"{feat} not in {features}")

    feat_idx   = features.index(feat)
    data       = X_full[feat_idx]
    _t_end     = n_t if t_end is None else min(t_end, n_t)
    frame_list = list(range(t_start, _t_end, step))
    cmap       = CMAPS.get(feat, 'viridis')
    cb_label   = LABELS.get(feat, feat)

    stride = max(1, len(frame_list) // 20)
    sample = data[:, frame_list[::stride]]
    finite = np.isfinite(sample)
    vmin   = np.percentile(sample[finite], p_low)  if finite.any() else 0.0
    vmax   = np.percentile(sample[finite], p_high) if finite.any() else 1.0
    del sample
    levels = np.linspace(vmin, vmax, 60)

    fig   = plt.figure(figsize=(_fig_w, _fig_h), dpi=150)
    ax    = fig.add_axes(ax_rect)
    cb_ax = fig.add_axes(cb_rect)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    cb = fig.colorbar(sm, cax=cb_ax)
    cb.set_label(cb_label, fontsize=FS, rotation=90, labelpad=4)
    cb.ax.tick_params(labelsize=FS - 1)

    def draw_frame(t_idx):
        ax.clear()
        field = data[:, t_idx]
        lv = levels if not dynamic_clim else np.linspace(
            np.percentile(field, p_low), np.percentile(field, p_high), 60)
        ax.tricontourf(triang_cm, field, levels=lv, cmap=cmap, extend='both')
        ax.set_xlim([0, r_max_cm])
        ax.set_ylim([z_min_cm, z_max_cm])
        ax.set_aspect('equal')
        ax.set_xlabel('r  [cm]', fontsize=FS)
        ax.set_ylabel('z  [cm]', fontsize=FS)
        ax.tick_params(labelsize=FS - 1)
        for spine in ax.spines.values():
            spine.set_linewidth(0.5)
        ax.set_title(f't = {t_idx * DT_SAVE * 1e3:.1f} ms', fontsize=FS)

    draw_frame(frame_list[0])
    ani = animation.FuncAnimation(fig, lambda i: draw_frame(frame_list[i]),
                                  frames=len(frame_list), interval=100, blit=False)
    plt.close(fig)
    return ani

HTML(animate_feature('T', step=20).to_jshtml())

# 2: PCA on a single snapshot

PCA can be applied to a **single snapshot** to explore the relationships between the thermochemical variables across space.

At a given time $t$, we build the data matrix:

$$\mathbf{D} \in \mathbb{R}^{n_s \times n_f}$$

where each row is a spatial point and each column is a feature (T, O₂, …, N₂).

PCA will find the directions of maximum variance in the $n_f$-dimensional feature space.

<img src="https://i.ytimg.com/vi/fsTSLXhz4uQ/hq720.jpg" width="400">


Before applying PCA, we center and scale the dataset so that each feature has zero mean and unit variance. Without this step, temperature, spanning hundreds of Kelvin, would not be comparable with species mass fractions, which live in $[0, 1]$, and the principal components would reflect units rather than physics.

- Center and scale the dataset:
\begin{equation*}
\mathbf{D}_0 = \frac{\mathbf{D} - \mathbf{D}_{\mu}}{\mathbf{D}_{\sigma}}
\end{equation*}

In [ ]:
# Build D from the last snapshot: (n_s, n_f)
t_pca = -1
D = X_full[:, :, t_pca].T          # (n_s, n_f)
print(f'D shape: {D.shape}  (n_s, n_f)')

# To do: Center and scale each column (feature)
# Hints:
# - Create two matrices D_m and D_s using np.mean() and np.std()

D_mu    = ...           # (n_f,)
D_sigma = ...           # (n_f,)
D0      = ...           # (n_s, n_f)

print(f'\nD0 mean : {D0.mean(axis=0).round(4)}')
print(f'D0 std  : {D0.std(axis=0).round(4)}')



## PCA algorithm

To find the PCs, we need to follow these steps:

- Compute the covariance matrix:
\begin{equation*}
\mathbf{K} = \mathbf{D}_0^T\mathbf{D}_0
\end{equation*}

- Compute the eigendecomposition of the covariance matrix:
\begin{equation*}
\mathbf{K} = \mathbf{\Psi} \mathbf{L} \mathbf{\Psi}^T
\end{equation*}

- Sort the eigenvectors in descending order of eigenvalues.

- Transform the scaled dataset:
\begin{equation*}
\mathbf{Z} = \mathbf{D}_0 \mathbf{\Psi}
\end{equation*}

- The original dataset can be reconstructed as:

\begin{equation*}
\mathbf{D} = (\mathbf{Z} \mathbf{\Psi}^T)\mathbf{D}_{\sigma} + \mathbf{D}_{\mu}
\end{equation*}


In [ ]:
# To do: compute the covariance matrix

# Hint:
# - Use the function np.transpose() or simply the .T method to transpose the matrix

K = ...                            # (n_f, n_f)
print(K.shape)

In [ ]:
# To do: compute the eigendecomposition of the covariance matrix and sort the eigenvalues and eigenvectors in decreasing order

# Hint:
# - Use the function np.linalg.eigh() to compute the eigendecomposition (it returns eigenvalues in ascending order)
# - Use np.argsort()[::-1] to sort the eigenvectors
eigvals, Psi = ...
idx     = ...
eigvals = ...
Psi     = ...

print(f'Eigenvalues: {eigvals}')

In [ ]:
# To do: compute the transformed dataset (projections of the data onto the PCs)
Z = ...
print(Z.shape)
print(D0.shape)

They project in the transformed space still have the dimension of the original flame so we can plot them:

In [ ]:
# Visualise first 4 PC scores
fig, axes = plt.subplots(1, 4, figsize=(12, 10), constrained_layout=True)

for i, (ax, pc) in enumerate(zip(axes, [0, 1, 2, 3])):
    field = Z[:, pc]
    cf = ax.tricontourf(triang_cm, field, levels=60, cmap='RdBu_r')
    cb = fig.colorbar(cf, ax=ax, shrink=0.5, pad=0.03)
    cb.set_label('Score  [-]', fontsize=9)
    cb.ax.tick_params(labelsize=7)

    ax.set_xlim([0, r_max_cm])
    ax.set_ylim([z_min_cm, z_max_cm])
    ax.set_aspect('equal')

    ax.set_xlabel('r  [cm]', fontsize=10)
    ax.tick_params(labelsize=8)
    ax.set_title(f'PC {pc+1}', fontsize=11,
                 fontweight='bold', pad=6)

    if i == 0:
        ax.set_ylabel('z  [cm]', fontsize=10)
    else:
        ax.set_yticklabels([])

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
plt.show()

Each one of them is a linear combination of the original features, with coefficients given by the corresponding eigenvector (loadings).

In [ ]:
loadings = Psi[:, 0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(features, loadings, color='steelblue', edgecolor='k', linewidth=0.6)
ax.axhline(0, color='k', linewidth=0.8)
ax.set_ylabel('PC 1 eigenvector weight', fontsize=10)
ax.set_xlabel('Feature', fontsize=10)
ax.set_title('PC 1 — loadings', fontsize=11, fontweight='bold')
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', labelsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

PC1 has a large positive weight on O₂ and N₂ and large negative weights on T, H₂O, CO₂ and OH, it separates the fresh oxidiser stream from the hot combustion products.


## Explained variance and truncation

The relative variance captured by each mode is:

$$V(i) = \frac{l_i}{\sum_j l_j}$$

where $l_i = L_{ii}$ are the eigenvalues of $\mathbf{K}$, sorted in descending order.

We select $r$ modes such that $\sum_{i=1}^r V(i) \geq \text{threshold}$, reducing the problem from $n_f$ to $r$.

In [ ]:
# Explained variance
# To do: compute the explained variance ratio and its cumulative sum

# Hint:
# - Use np.sum() to compute the total variance
# - Use np.cumsum() to compute the running cumulative sum

expl_var     = ...
cum_expl_var = ...

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar(np.arange(1, n_f+1), expl_var * 100, color='cornflowerblue', edgecolor='k')
ax1.set_xlabel('PC index');  ax1.set_ylabel('Explained variance (%)')
ax1.set_title('PCA — explained variance per PC')
ax1.grid(axis='y', linestyle='--', alpha=0.5)

ax2.plot(np.arange(1, n_f+1), cum_expl_var * 100, marker='o', color='darkorange')
ax2.axhline(99, color='r', ls='--', lw=1.2, label='99% threshold')
ax2.set_xlabel('PC index');  ax2.set_ylabel('Cumulative variance (%)')
ax2.set_title('PCA — cumulative variance')
ax2.legend();  ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f'Explained variance : {[f"{v:.1f}%" for v in expl_var * 100]}')
print(f'Cumulative variance : {[f"{v:.1f}%" for v in cum_expl_var * 100]}')

The analysis shows that only 4 modes capture 98.4% of the variance, the remaining modes contribute negligibly and can be discarded, reducing the data dimensionality from $n_f = 10$ to $r = 4$ and saving 60% of memory while preserving accuracy.

## PCA Flame Reconstruction

Having identified the principal components and their explained variance, we now reconstruct the original snapshot using a truncated set of modes.

For $r$ retained modes, the reconstruction in the scaled space is:

$$\tilde{\mathbf{D}}_{0} = \mathbf{Z}_{r}\,\mathbf{\Psi}_{r}^T$$

and in physical units:

$$\tilde{\mathbf{D}} = \tilde{\mathbf{D}}_{0} \cdot \mathbf{D}_{\sigma} + \mathbf{D}_{\mu}$$

We compare the original temperature field against reconstructions using **1, 2, 4, 6, and 10 modes** to visualize how quickly PCA captures the dominant flame structure.

In [ ]:
n_modes_list = [1, 2, 4, 6, 10]
feat_name    = 'T'
feat_idx     = features.index(feat_name)

T_orig = D[:, feat_idx]
vmin, vmax = T_orig.min(), T_orig.max()
levels = np.linspace(vmin, vmax, 60)

fig, axes = plt.subplots(1, len(n_modes_list) + 1, figsize=(14, 8))
fig.subplots_adjust(wspace=0.05, left=0.05, right=0.97)

# ── Original ──────────────────────────────────────────────────────────
ax = axes[0]
cf = ax.tricontourf(triang_cm, T_orig, levels=levels, cmap=CMAPS[feat_name])
cb = fig.colorbar(cf, ax=ax, shrink=0.5, pad=0.02)
cb.set_label('T [K]', fontsize=9);  cb.ax.tick_params(labelsize=7)
ax.set_xlim([0, r_max_cm]);  ax.set_ylim([z_min_cm, z_max_cm])
ax.set_aspect('equal')
ax.set_xlabel('r  [cm]', fontsize=9);  ax.set_ylabel('z  [cm]', fontsize=9)
ax.tick_params(labelsize=8)
ax.set_title('Original', fontsize=11, fontweight='bold')
for spine in ax.spines.values():
    spine.set_linewidth(0.8)

# ── Reconstructions ────────────────────────────────────────────────────
errors = []
for j, r_trunc in enumerate(n_modes_list):
    D_rec = (Z[:, :r_trunc] @ Psi[:, :r_trunc].T) * D_sigma + D_mu
    T_rec = D_rec[:, feat_idx]
    err   = np.linalg.norm(T_orig - T_rec) / np.linalg.norm(T_orig) * 100
    errors.append(err)

    ax = axes[j + 1]
    cf = ax.tricontourf(triang_cm, T_rec, levels=levels,
                        cmap=CMAPS[feat_name], extend='both')
    cb = fig.colorbar(cf, ax=ax, shrink=0.5, pad=0.02)
    cb.set_label('T [K]', fontsize=9);  cb.ax.tick_params(labelsize=7)
    ax.set_xlim([0, r_max_cm]);  ax.set_ylim([z_min_cm, z_max_cm])
    ax.set_aspect('equal')
    ax.set_xlabel('r  [cm]', fontsize=9);  ax.set_yticklabels([])
    ax.tick_params(labelsize=8)
    label = f'{r_trunc} mode{"s" if r_trunc > 1 else ""}'
    ax.set_title(f'{label}\n(err = {err:.2f}%)', fontsize=10, fontweight='bold')
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

plt.show()

print('Relative L2 reconstruction error (temperature):')
for r_trunc, err in zip(n_modes_list, errors):
    print(f'  {r_trunc:2d} mode{"s" if r_trunc > 1 else " "}: {err:.4f}%')

PC1 has a large positive weight on O₂ and large negative weights on T, H₂O, CO₂ and OH, it separates the fresh oxidiser stream from the hot combustion products.

# 3: Proper Orthogonal Decomposition (POD)

In Exercise 2, PCA was applied to a single snapshot to explore correlations between thermochemical variables across space, the decomposition was in the feature space ($n_f = 10$ dimensions).

POD extends this idea to the full time series, instead of one snapshot, we stack all $n_t = 2001$ snapshots and decompose in the temporal direction. This reveals the dominant spatial structures that recur across time, ranked by their energetic contribution.

POD is dimensionality reduction technique used to

*   Identify the most energetic structures in the flow.
*   Compress the data.

There are 3 ways to compute the POD decomposition:

* Eigendecomposition of the spatial correlation matrix (classic POD).
* Eigendecomposition of the temporal correlation matrix (snapshot POD).
* Singular value decomposition.

Today we use **Eigendecomposition of the temporal correlation matrix**.

## Scaling and Centering

As in PCA, we standardize the data before decomposing. Here $X_f \in \mathbb{R}^{n_s \times n_t}$ is the matrix of a single feature $f$ (e.g. temperature, or $Y_{\mathrm{OH}}$) over all spatial points and all snapshots. For each feature we compute the mean, the std and apply:

$$X_{0,f} = \frac{X_f - \mu_f}{\sigma_f}$$

This ensures that all features have mean $= 0$ and standard deviation $= 1$.

In [ ]:
# To do: Center and scale each feature (over all spatial points and all snapshots)
# Hints:
# - Loop over features and use np.mean() and np.std() on each feature slice

X0      = np.zeros_like(X_full, dtype=np.float32)
X_mu    = np.zeros(n_f)
X_sigma = np.zeros(n_f)

.
.
.

print(f'X0 mean : {X0.mean(axis=(1, 2)).round(4)}')
print(f'X0 std  : {X0.std(axis=(1, 2)).round(4)}')

## Data matrix

We first reshape the scaled tensor $\mathbf{X}_0 \in \mathbb{R}^{n_f \times n_s \times n_t}$ into a 2D **snapshot matrix** by stacking all features along the spatial axis:

$$\mathbf{D}_0  \in \mathbb{R}^{(n_f \cdot n_s) \times n_t}$$

Each column of $\mathbf{D}_0$ is a single snapshot containing all features at all spatial points.

In [ ]:
# Reshape X0:
D0 = X0.reshape(-1, n_t).astype(np.float64)
print(f'Snapshot matrix D0: {D0.shape}  (n_f * n_s, n_t)')

## POD algorithm (snapshot method)

When $n_f \cdot n_s \gg n_t$, it is more efficient to work with the **temporal correlation matrix**:

$$\mathbf{K} = \mathbf{D}_0^T \mathbf{D}_0 \in \mathbb{R}^{n_t \times n_t}$$

and compute its eigendecomposition, same steps as PCA, but now in the temporal direction:

$$\mathbf{K} = \mathbf{V} \mathbf{L} \mathbf{V}^T$$

The eigenvectors $\mathbf{V}$ are the principal components in the temporal space, and the eigenvalues $\mathbf{L}$ represent the energy content of each mode.

The singular values are obtained as:

$$\mathbf{S} = \sqrt{\mathbf{L}}$$

The spatial POD modes are found by transforming back to the spatial domain:

$$\mathbf{\Phi} = \mathbf{D}_0 \mathbf{V} \mathbf{S}^{-1}$$

The temporal coefficients are computed as:

$$\mathbf{A} = \mathbf{V} \mathbf{S}$$

giving the decomposition:

$$\mathbf{D}_0 = \mathbf{\Phi}\mathbf{A}^T$$

This mirrors the PCA reconstruction $\mathbf{D}_0 = \mathbf{Z}\mathbf{\Psi}^T$: in both cases, spatial structure $\times$ coefficients transposed.

In [ ]:
import time

def decompose(D):
    # To do: compute the POD using the snapshot method
    # Hint:
    # - Use np.linalg.eigh() and np.argsort()[::-1] to sort
    # - Use np.sqrt()


    K = ...
    eigvals, V = ...

    idx     = ...
    eigvals = ...
    V       = ...

    S   = ...
    Phi = ...
    A   = ...

    return Phi, A, S

t0 = time.time()
Phi, A, S = decompose(D0)
print(f'Decomposition done in {time.time()-t0:.1f} s')
print(f'Phi: {Phi.shape}   A: {A.shape}   S: {S.shape}')

In [ ]:
def plot_pod_energy(S, target=99.0, max_modes=50):
    energy     = S**2
    total      = energy.sum()
    energy_pct = 100.0 * energy / total
    cum_energy = np.cumsum(energy_pct)

    r_target = np.sum(cum_energy < target) + 1
    n_show   = min(max_modes, len(energy))
    modes    = np.arange(1, n_show + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.bar(modes, energy_pct[:n_show], color='cornflowerblue', edgecolor='k')
    ax1.set_xlabel('Mode index')
    ax1.set_ylabel('Energy per mode (%)')
    ax1.set_title('POD — energy per mode')
    ax1.grid(axis='y', linestyle='--', alpha=0.5)

    ax2.plot(modes, cum_energy[:n_show], marker='o', color='darkorange', markersize=4)
    ax2.axhline(target, color='r', ls='--', lw=1.5, label=f'{target:.0f}% threshold')
    if r_target <= n_show:
        ax2.axvline(r_target, color='gray', ls='--', lw=1.2)
        ax2.text(r_target + 1, target - 5, f'{r_target} modes', fontsize=10)
    ax2.set_xlabel('Mode index')
    ax2.set_ylabel('Cumulative energy (%)')
    ax2.set_title('POD — cumulative energy')
    ax2.legend()
    ax2.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()
    print(f'→ {r_target} modes needed to capture {target:.1f}% of the variance')
    return r_target

r = plot_pod_energy(S, target=98.0)

Phi_r, Ar, Sr = Phi[:, :r], A[:, :r], S[:r]
print(f'Phi: {Phi.shape}   A: {A.shape}')
print(f'Phi_r: {Phi_r.shape}   Ar: {Ar.shape}')

With $r = 6$, the storage drops from $n_f \cdot n_s \times n_t = 136{,}000 \times 2001 \approx 272\text{M}$ values to $(136{,}000 + 2001) \times 6 \approx 828\text{K}$, a compression of roughly 329×, while retaining 98% of the energy.

In [ ]:
def plot_pod_coefficients(A, num_modes=5, t_indices=None):
    """Plot temporal evolution of the first POD coefficients."""
    n_t, _ = A.shape
    x_axis = t_indices if t_indices is not None else np.arange(n_t)

    fig, ax = plt.subplots(figsize=(9, 4))
    for i in range(min(num_modes, A.shape[1])):
        ax.scatter(x_axis, A[:, i], label=f'Mode {i+1}', s=5, alpha=0.6)
    ax.set_xlabel('Snapshot index')
    ax.set_ylabel('Coefficient')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_pod_coefficients(A, num_modes=5)

## POD Flame Reconstruction

Having computed the POD modes and their energy content, we now reconstruct a snapshot using a truncated set of modes.

For $r$ retained modes, the reconstruction of snapshot $t$ in the scaled space is:

$$\tilde{\mathbf{d}}_t = \boldsymbol{\Phi}_r\,\mathbf{a}_{r,t}$$

where $\mathbf{a}_{r,t} = \mathbf{A}[t,\,:r]$ is the vector of $r$ temporal coefficients at time $t$.

To recover physical units for feature $f$:

$$\tilde{X}_f = \tilde{d}_f \cdot \sigma_f + \mu_f$$

We compare the original temperature field against reconstructions using **1, 2, 5, 10, and 20 modes** to visualize how quickly POD captures the dominant flame dynamics.

In [ ]:
n_modes_list = [1, 2, 5, 10, 20]
feat_name    = 'T'
feat_idx     = features.index(feat_name)
t_snap       = n_t // 2          # middle snapshot

# Original temperature at t_snap
T_orig = X_full[feat_idx, :, t_snap]
vmin, vmax = T_orig.min(), T_orig.max()
levels = np.linspace(vmin, vmax, 60)

fig, axes = plt.subplots(1, len(n_modes_list) + 1, figsize=(14, 8))
fig.subplots_adjust(wspace=0.05, left=0.05, right=0.97)

# ── Original ──────────────────────────────────────────────────────────
ax = axes[0]
cf = ax.tricontourf(triang_cm, T_orig, levels=levels, cmap=CMAPS[feat_name])
cb = fig.colorbar(cf, ax=ax, shrink=0.5, pad=0.02)
cb.set_label('T [K]', fontsize=9);  cb.ax.tick_params(labelsize=7)
ax.set_xlim([0, r_max_cm]);  ax.set_ylim([z_min_cm, z_max_cm])
ax.set_aspect('equal')
ax.set_xlabel('r  [cm]', fontsize=9);  ax.set_ylabel('z  [cm]', fontsize=9)
ax.tick_params(labelsize=8)
ax.set_title('Original', fontsize=11, fontweight='bold')
for spine in ax.spines.values():
    spine.set_linewidth(0.8)

# ── Reconstructions ────────────────────────────────────────────────────
# To do: reconstruct the snapshot at t_snap for each number of modes
# Hints:
# - d_rec = Phi[:, :r_trunc] @ A[t_snap, :r_trunc]   shape: (n_f * n_s,)
# - reshape to (n_f, n_s) then reverse scaling:
#   T_rec = d_rec.reshape(n_f, n_s)[feat_idx] * X_sigma[feat_idx] + X_mu[feat_idx]
errors = []
for j, r_trunc in enumerate(n_modes_list):
    d_rec = ...              # (n_f * n_s,)
    T_rec = ...              # (n_s,)

    err = np.linalg.norm(T_orig - T_rec) / np.linalg.norm(T_orig) * 100
    errors.append(err)

    ax = axes[j + 1]
    cf = ax.tricontourf(triang_cm, T_rec, levels=levels,
                        cmap=CMAPS[feat_name], extend='both')
    cb = fig.colorbar(cf, ax=ax, shrink=0.5, pad=0.02)
    cb.set_label('T [K]', fontsize=9);  cb.ax.tick_params(labelsize=7)
    ax.set_xlim([0, r_max_cm]);  ax.set_ylim([z_min_cm, z_max_cm])
    ax.set_aspect('equal')
    ax.set_xlabel('r  [cm]', fontsize=9);  ax.set_yticklabels([])
    ax.tick_params(labelsize=8)
    label = f'{r_trunc} mode{"s" if r_trunc > 1 else ""}'
    ax.set_title(f'{label}\n(err = {err:.2f}%)', fontsize=10, fontweight='bold')
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

plt.show()

print('Relative L2 reconstruction error (temperature):')
for r_trunc, err in zip(n_modes_list, errors):
    print(f'  {r_trunc:2d} mode{"s" if r_trunc > 1 else " "}: {err:.4f}%')

# Extra: POD Super-Resolution via Linear Interpolation

Once the spatial modes are known, the full field at any time is determined by just $r$ temporal coefficients. This means we can reconstruct missing snapshots by interpolating the coefficients, not the full field.

- **Subsample**: keep 1 snapshot every 5
- **Interpolate**: linearly interpolate the POD coefficients to reconstruct all intermediate snapshots
- **Reconstruct**: apply the spatial modes and reverse the scaling to get back physical fields
- **Evaluate**: compute the error only at the interpolated snapshots

In [ ]:
# To do: keep 1 snapshot every 5, linearly interpolate the POD coefficients
#         to reconstruct all intermediate snapshots, then check the error.
# Hints:
# - use np.arange(0, n_t, 5) to get the sparse time indices
# - use np.interp(t_full, t_sparse, values) to interpolate each coefficient
# - loop over the r modes to interpolate each coefficient independently